# 🎬 — Movie Recommendation ML Pipeline (MovieLens Edition)

**Dataset:** MovieLens — `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`

**Pipeline steps:**
1. Install & import dependencies
2. Load and merge the four MovieLens CSVs into `MovieData`
3. Collapse per-user rows → one row per movie; clean genres, tags, and title
4. Build TF-IDF soup (genres + user tags + title)
5. Cosine similarity → top-K neighbour index
6. K-Means clustering (TF-IDF + numeric features: avg_rating, rating_count, year)
7. Silhouette scores per movie
8. Persist all artifacts to disk
9. Serving class (`CineScopeRecommender`)
10. Interactive demo

> 📌 **Expected columns in `MovieData`:** `userId`, `movieId`, `rating`, `timestamp`,
> `title` (with year, e.g. `"Toy Story (1995)"`), `genres` (pipe-separated),
> `imdbId`, `tmdbId`, `tag` (aggregated, pipe-separated)

## ① Install & Import Dependencies

In [1]:
!pip install -q scikit-learn pandas numpy scipy joblib pyarrow tqdm

In [2]:
import json
import os
import re
import shutil
import string
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display, HTML
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_samples
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Artifact output directory ──────────────────────────────────────────────────
ARTIFACTS_DIR = Path('backend/artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# ── Pipeline config ────────────────────────────────────────────────────────────
N_CLUSTERS   = 7      # number of K-Means clusters
TOP_K        = 100    # cosine neighbours to store per movie
CHUNK_SIZE   = 200    # rows per chunk for cosine computation
RANDOM_STATE = 42
MIN_RATINGS  = 5      # drop movies with fewer than this many ratings (noise filter)

print('✅ All imports OK')
print(f'   Artifacts will be saved to: {ARTIFACTS_DIR.resolve()}')

✅ All imports OK
   Artifacts will be saved to: /Users/ayoadeabraham/PycharmProjects/silver-screen/backend/artifacts


/opt/anaconda3/envs/beans/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ② Load Dataset

Update the four paths below to point to your MovieLens CSV files.
The files are merged exactly as in the original notebook to produce `MovieData`.

In [3]:
# ── Update these paths to your MovieLens data files ───────────────────────────
# In Google Colab you can upload files and use the path shown after upload,
# or mount Google Drive and point to the files there.
#
# Example (Google Drive):
#   from google.colab import drive
#   drive.mount('/content/drive')
#   DATA_DIR = Path('/content/drive/MyDrive/movielens')
#
# Example (local Colab upload via files.upload()):
#   from google.colab import files
#   files.upload()  # upload all four CSVs, then set DATA_DIR = Path('.')

DATA_DIR = Path('/Users/ayoadeabraham/PycharmProjects/silver-screen/src/data')   # ← change to your data directory

movies_csv  = DATA_DIR / 'movies.csv'
ratings_csv = DATA_DIR / 'ratings.csv'
tags_csv    = DATA_DIR / 'tags.csv'
links_csv   = DATA_DIR / 'links.csv'

# ── Load ───────────────────────────────────────────────────────────────────────
movies  = pd.read_csv(movies_csv)
ratings = pd.read_csv(ratings_csv)
tags    = pd.read_csv(tags_csv)
links   = pd.read_csv(links_csv)

# ── Merge exactly as in the original notebook ─────────────────────────────────
df = ratings.merge(movies, on='movieId', how='left')
df = df.merge(links,  on='movieId', how='left')

tags_agg = (
    tags
    .groupby('movieId')['tag']
    .apply(lambda x: ' | '.join(x.astype(str)))
    .reset_index()
)
df = df.merge(tags_agg, on='movieId', how='left')

MovieData = df

print(f'✅ MovieData loaded: {len(MovieData):,} rows × {MovieData.shape[1]} columns')
print(f'   Unique movies : {MovieData["movieId"].nunique():,}')
print(f'   Unique users  : {MovieData["userId"].nunique():,}')
print(f'   Columns       : {MovieData.columns.tolist()}')
display(MovieData.head(5))

✅ MovieData loaded: 32,000,204 rows × 9 columns
   Unique movies : 84,432
   Unique users  : 200,948
   Columns       : ['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres', 'imdbId', 'tmdbId', 'tag']


,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId,tag
0,1,17,4.0,944249077,Sense and Sensibility (1995),Drama|Romance,114388,4584.0,boring | nothing happens | 18th century | clas...
1,1,25,1.0,944250228,Leaving Las Vegas (1995),Drama|Romance,113627,451.0,based on a book | depressing | drama | Nicolas...
2,1,29,2.0,943230976,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,112682,902.0,dystopia | Jean-Pierre Jeunet | surreal | dark...
3,1,30,5.0,944249077,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Crime|Drama,115012,37557.0,cabaret | diva | mistress | servant | shanghai...
4,1,32,5.0,943228858,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,114746,63.0,atmospheric | Brad Pitt | Bruce Willis | desig...


## ③ Data Cleaning & Feature Construction

**Key differences from TMDb:**
- `genres` is a pipe-separated string (`"Action|Crime|Drama"`) — split on `|`
- `tag` is aggregated user tags separated by ` | ` — split and rejoin for the soup
- No `overview`, `cast`, `crew`, or `keywords` columns exist
- Titles include the release year in parentheses (`"Toy Story (1995)"`) — extract and strip it
- MovieData has one row per user rating — collapse to **one row per movie**
  and compute `avg_rating` and `rating_count` as the numeric signal

In [4]:
# ── Helpers ────────────────────────────────────────────────────────────────────

def parse_genres(genres_str: str) -> list:
    """Split pipe-separated genres string into a list of genre names.
    Input : 'Action|Crime|Drama'
    Output: ['Action', 'Crime', 'Drama']
    """
    if not isinstance(genres_str, str) or not genres_str.strip():
        return []
    return [g.strip() for g in genres_str.split('|') if g.strip() and g.strip() != '(no genres listed)']

def parse_tags(tag_str) -> list:
    """Split pipe-separated aggregated tag string into individual tags.
    Input : 'atmospheric | thought-provoking | sci-fi'
    Output: ['atmospheric', 'thought-provoking', 'sci-fi']
    """
    if not isinstance(tag_str, str) or not tag_str.strip():
        return []
    return [t.strip().lower() for t in tag_str.split('|') if t.strip()]

def extract_year(title: str):
    """Extract the 4-digit year from a title like 'Toy Story (1995)'.
    Returns the year as int, or 0 if not found.
    """
    m = re.search(r'\((\d{4})\)\s*$', str(title))
    return int(m.group(1)) if m else 0

def clean_title(title: str) -> str:
    """Remove the trailing '(year)' suffix and lowercase the title."""
    title = re.sub(r'\(\d{4}\)\s*$', '', str(title)).strip()
    title = title.lower().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', title).strip()

def tokens_to_str(token_list: list) -> str:
    """Join tokens into a compact lowercase string (no spaces within tokens)."""
    return ' '.join(t.lower().replace(' ', '') for t in token_list if t)

print('✅ Helpers defined')

✅ Helpers defined


In [5]:
# ── Collapse MovieData: one row per movie ─────────────────────────────────────
# MovieData has one row per (userId, movieId) rating.
# We aggregate ratings to get avg_rating and rating_count per movie,
# then join back to the unique movie metadata.

# 1. Compute per-movie rating stats from the full MovieData
rating_stats = (
    MovieData
    .groupby('movieId')['rating']
    .agg(avg_rating='mean', rating_count='count')
    .reset_index()
)

# 2. Unique movie metadata (one row per movieId)
movie_meta = (
    MovieData[['movieId', 'title', 'genres', 'imdbId', 'tmdbId', 'tag']]
    .drop_duplicates(subset='movieId', keep='first')
    .reset_index(drop=True)
)

# 3. Join stats onto metadata
df = movie_meta.merge(rating_stats, on='movieId', how='left')
df['avg_rating']   = df['avg_rating'].fillna(0.0).round(4)
df['rating_count'] = df['rating_count'].fillna(0).astype(int)

# 4. Filter low-signal movies
df = df[df['rating_count'] >= MIN_RATINGS].reset_index(drop=True)

# 5. Parse and clean columns
df['genres_list'] = df['genres'].apply(parse_genres)
df['tags_list']   = df['tag'].apply(parse_tags)
df['year']        = df['title'].apply(extract_year).astype(float)
df['clean_title'] = df['title'].apply(clean_title)

# 6. Stable integer movie_id (0-indexed row position)
df = df.reset_index(drop=True)
df['movie_id'] = df.index

print(f'✅ Per-movie dataset: {len(df):,} movies (after {MIN_RATINGS}+ ratings filter)')
print(f'   avg_rating range  : {df["avg_rating"].min():.2f} – {df["avg_rating"].max():.2f}')
print(f'   rating_count range: {df["rating_count"].min()} – {df["rating_count"].max():,}')
print(f'   year range        : {int(df["year"].min())} – {int(df["year"].max())}')
print()
print(df[['movieId', 'title', 'genres_list', 'avg_rating', 'rating_count', 'year']].head(6))

✅ Per-movie dataset: 43,884 movies (after 5+ ratings filter)
   avg_rating range  : 0.50 – 4.86
   rating_count range: 5 – 102,929
   year range        : 0 – 2023

   movieId                                              title  \
0       17                       Sense and Sensibility (1995)   
1       25                           Leaving Las Vegas (1995)   
2       29  City of Lost Children, The (Cité des enfants p...   
3       30  Shanghai Triad (Yao a yao yao dao waipo qiao) ...   
4       32          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   
5       34                                        Babe (1995)   

                                    genres_list  avg_rating  rating_count  \
0                              [Drama, Romance]      3.9451         22251   
1                              [Drama, Romance]      3.6762         22525   
2  [Adventure, Drama, Fantasy, Mystery, Sci-Fi]      3.9299          9413   
3                                [Crime, Drama]      3.6296          130

## ④ TF-IDF Vectorization

**Soup composition for MovieLens:**
- `genres` — the strongest genre signal (repeated for emphasis)
- `tags_list` — community-contributed descriptors (very high signal)
- `clean_title` — the film title without year

There is no overview, cast, or crew in this dataset, so genre + community tags
carry the full semantic weight.

In [6]:
# ── Build soup ────────────────────────────────────────────────────────────────
def build_soup(row) -> str:
    """
    Combine genres (×2 for emphasis), user tags, and clean title into
    a single text blob for TF-IDF vectorisation.
    """
    genre_str = tokens_to_str(row['genres_list'])
    tag_str   = tokens_to_str(row['tags_list'])
    title_str = row['clean_title']
    # Repeat genres twice to give them stronger relative weight
    parts = [genre_str, genre_str, tag_str, title_str]
    return ' '.join(p for p in parts if p)

df['soup_text'] = df.apply(build_soup, axis=1)

print('Soup sample:')
for i in range(3):
    print(f'  [{df.iloc[i]["title"]}]')
    print(f'   {df.iloc[i]["soup_text"][:140]}')
    print()

Soup sample:
  [Sense and Sensibility (1995)]
   drama romance drama romance boring nothinghappens 18thcentury classic emotional british janeausten romance janeausten periodpiece romance se

  [Leaving Las Vegas (1995)]
   drama romance drama romance basedonabook depressing drama nicolascage nudity(topless) oscar(bestactor) prostitution addiction alcoholism atm

  [City of Lost Children, The (Cité des enfants perdus, La) (1995)]
   adventure drama fantasy mystery sci-fi adventure drama fantasy mystery sci-fi dystopia jean-pierrejeunet surreal darkfantasy jean-pierrejeun



In [7]:
# ── Fit TF-IDF ────────────────────────────────────────────────────────────────
print('Fitting TF-IDF vectorizer...')

tfidf        = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.95)
tfidf_matrix = tfidf.fit_transform(df['soup_text'])

# Index maps: title ↔ row index ↔ movie_id
title_to_index    = {row['title']: i for i, row in df.iterrows()}
index_to_movie_id = {i: int(row['movie_id']) for i, row in df.iterrows()}
movie_id_to_index = {int(row['movie_id']): i for i, row in df.iterrows()}

print(f'✅ TF-IDF matrix shape : {tfidf_matrix.shape}')
print(f'   Vocabulary size     : {len(tfidf.vocabulary_):,} terms')
sparsity = 1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
print(f'   Sparsity            : {sparsity:.1%}')

Fitting TF-IDF vectorizer...
✅ TF-IDF matrix shape : (43884, 60416)
   Vocabulary size     : 60,416 terms
   Sparsity            : 100.0%


## ⑤ Cosine Similarity → Top-K Neighbour Index

In [8]:
n = tfidf_matrix.shape[0]
cosine_topk_index = {}   # {movie_index: [(neighbor_index, score), ...]}

print(f'Building cosine top-{TOP_K} index for {n:,} movies (chunk_size={CHUNK_SIZE})...')

for start in tqdm(range(0, n, CHUNK_SIZE), desc='Cosine chunks'):
    end  = min(start + CHUNK_SIZE, n)
    sims = linear_kernel(tfidf_matrix[start:end], tfidf_matrix)   # (chunk, n)

    k = min(n, TOP_K + 1)
    for local_i, row_sims in enumerate(sims):
        global_i = start + local_i
        top_idx  = np.argpartition(row_sims, -k)[-k:]
        top_idx  = top_idx[np.argsort(row_sims[top_idx])[::-1]]
        cosine_topk_index[global_i] = [
            (int(j), float(row_sims[j]))
            for j in top_idx if j != global_i
        ][:TOP_K]

print(f'✅ Cosine index built: {len(cosine_topk_index):,} entries')

# Preview top-5 neighbours for the first movie
sample_title = df.iloc[0]['title']
print(f'\nTop-5 neighbours for "{sample_title}":')
for nb_idx, score in cosine_topk_index[0][:5]:
    print(f'  [{score:.3f}]  {df.iloc[nb_idx]["title"]}')

Building cosine top-100 index for 43,884 movies (chunk_size=200)...


Cosine chunks: 100%|██████████| 220/220 [00:39<00:00,  5.58it/s]

✅ Cosine index built: 43,884 entries

Top-5 neighbours for "Sense and Sensibility (1995)":
  [0.668]  Emma (1996)
  [0.648]  Persuasion (1995)
  [0.622]  Northanger Abbey (2007)
  [0.598]  Persuasion (2007)
  [0.593]  Pride and Prejudice (1940)


## ⑥ K-Means Clustering (TF-IDF + Numeric Features)

**Numeric features for MovieLens:**
- `avg_rating` — community-weighted quality signal
- `rating_count` — popularity / how widely watched
- `year` — release era (clusters tend to group by decade)

These replace the TMDb fields (`popularity`, `vote_count`, `vote_average`, `runtime`).

In [9]:
print('Building combined feature matrix (TF-IDF + numeric)...')

# ── Scale numeric features ─────────────────────────────────────────────────────
num_cols   = ['avg_rating', 'rating_count', 'year']
num_feats  = df[num_cols].fillna(0).values
scaler     = StandardScaler()
num_scaled = scaler.fit_transform(num_feats)

# ── Combine: sparse TF-IDF + dense numeric ─────────────────────────────────────
X = sp.hstack([tfidf_matrix, sp.csr_matrix(num_scaled)])
print(f'   Combined feature matrix: {X.shape}')

# ── Fit K-Means ────────────────────────────────────────────────────────────────
print(f'\nFitting KMeans(n_clusters={N_CLUSTERS})...')
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X)
df['cluster'] = kmeans.labels_

# ── Cluster composition preview ────────────────────────────────────────────────
print(f'\n✅ KMeans fitted')
for c, cnt in df['cluster'].value_counts().sort_index().items():
    sample_titles = df[df['cluster'] == c]['title'].tolist()[:3]
    print(f'  Cluster {c} ({cnt:,} movies): {sample_titles}')

Building combined feature matrix (TF-IDF + numeric)...
   Combined feature matrix: (43884, 60419)

Fitting KMeans(n_clusters=7)...

✅ KMeans fitted
  Cluster 0 (15,706 movies): ['Foxfire (1996)', 'Married to the Mob (1988)', 'Airport (1970)']
  Cluster 1 (4,060 movies): ['Showgirls (1995)', 'Superman IV: The Quest for Peace (1987)', 'Speed 2: Cruise Control (1997)']
  Cluster 2 (137 movies): ['Twelve Monkeys (a.k.a. 12 Monkeys) (1995)', 'Babe (1995)', 'Braveheart (1995)']
  Cluster 3 (165 movies): ['The Favourite', 'Ava', 'Ready Player One']
  Cluster 4 (907 movies): ['Sense and Sensibility (1995)', 'Leaving Las Vegas (1995)', 'City of Lost Children, The (Cité des enfants perdus, La) (1995)']
  Cluster 5 (11,844 movies): ['Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)', 'White Balloon, The (Badkonake sefid) (1995)', 'Living in Oblivion (1995)']
  Cluster 6 (11,065 movies): ['Doom Generation, The (1995)', 'Jakob the Liar (1999)', 'Ready to Wear (Pret-A-Porter) (1994)']


## ⑦ Silhouette Scores (Per Movie)

In [10]:
print('Computing per-movie silhouette scores...')

n_movies = X.shape[0]
MAX_SIL  = 5000   # cap sample for large datasets to keep runtime reasonable

if n_movies <= MAX_SIL:
    X_dense    = X.toarray()
    labels_sil = kmeans.labels_
else:
    print(f'  Large dataset ({n_movies:,} movies): sampling {MAX_SIL:,} rows for silhouette')
    rng        = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(n_movies, MAX_SIL, replace=False)
    X_dense    = X[sample_idx].toarray()
    labels_sil = kmeans.labels_[sample_idx]

sil_scores = silhouette_samples(X_dense, labels_sil)

# Map scores back to full df (unseen rows get 0.0)
df['silhouette'] = 0.0
if n_movies <= MAX_SIL:
    df['silhouette'] = sil_scores
else:
    df.loc[sample_idx, 'silhouette'] = sil_scores

print(f'✅ Silhouette scores computed')
print(f'   Mean: {sil_scores.mean():.4f}  |  Min: {sil_scores.min():.4f}  |  Max: {sil_scores.max():.4f}')

print('\n🏆 Best-fit movie per cluster (highest silhouette):')
for c in sorted(df['cluster'].unique()):
    g    = df[df['cluster'] == c]
    best = g.loc[g['silhouette'].idxmax()]
    print(f'  Cluster {c}: "{best["title"]}" (silhouette={best["silhouette"]:.4f})')

Computing per-movie silhouette scores...
  Large dataset (43,884 movies): sampling 5,000 rows for silhouette
✅ Silhouette scores computed
   Mean: 0.1140  |  Min: -0.0252  |  Max: 0.8959

🏆 Best-fit movie per cluster (highest silhouette):
  Cluster 0: "Bopha! (1993)" (silhouette=0.1656)
  Cluster 1: "Komodo vs. Cobra (2005)" (silhouette=0.3458)
  Cluster 2: "Twelve Monkeys (a.k.a. 12 Monkeys) (1995)" (silhouette=0.6766)
  Cluster 3: "Falling Skies" (silhouette=0.8959)
  Cluster 4: "10 Things I Hate About You (1999)" (silhouette=0.5213)
  Cluster 5: "Time Indefinite (1993)" (silhouette=0.2545)
  Cluster 6: "Daisy-Head Mayzie (1995)" (silhouette=0.2905)


## ⑧ Persist All Artifacts

In [11]:
import sys
!{sys.executable} -m pip install -U pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 17.8 MB/s  0:00:01m0:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.0
    Uninstalling pyarrow-23.0.0:
      Successfully uninstalled pyarrow-23.0.0


In [12]:
TMP_DIR = Path('artifacts_tmp')
TMP_DIR.mkdir(exist_ok=True)

def save_artifacts(out_dir: Path):
    # 1. TF-IDF vectorizer
    joblib.dump(tfidf, out_dir / 'tfidf_vectorizer.pkl')

    # 2. TF-IDF matrix (sparse)
    sp.save_npz(str(out_dir / 'tfidf_matrix.npz'), tfidf_matrix)

    # 3. Cosine top-K index
    with open(out_dir / 'cosine_topk_index.json', 'w') as f:
        json.dump({str(k): v for k, v in cosine_topk_index.items()}, f)

    # 4. K-Means model + numeric scaler
    joblib.dump(kmeans, out_dir / 'kmeans.pkl')
    joblib.dump(scaler, out_dir / 'scaler.pkl')

    # 5. Cluster + silhouette sidecar
    df[['movie_id', 'movieId', 'title', 'cluster', 'silhouette']].to_parquet(
        out_dir / 'movie_ml.parquet', index=False)

    # 6. Index maps
    with open(out_dir / 'title_to_index.json', 'w') as f:
        json.dump(title_to_index, f, ensure_ascii=False)
    with open(out_dir / 'index_to_movie_id.json', 'w') as f:
        json.dump({str(k): v for k, v in index_to_movie_id.items()}, f)

    # 7. Full movie metadata
    meta_cols = [
        'movie_id', 'movieId', 'title', 'clean_title', 'year',
        'genres_list', 'tags_list',
        'avg_rating', 'rating_count',
        'imdbId', 'tmdbId',
        'cluster', 'silhouette', 'soup_text',
    ]
    existing = [c for c in meta_cols if c in df.columns]
    df[existing].to_parquet(out_dir / 'movies_metadata.parquet', index=False)

print('Writing artifacts to temp directory...')
save_artifacts(TMP_DIR)

# Atomic swap
if ARTIFACTS_DIR.exists():
    shutil.rmtree(ARTIFACTS_DIR)
TMP_DIR.rename(ARTIFACTS_DIR)

print(f'\n✅ Artifacts saved to {ARTIFACTS_DIR.resolve()}')
for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f'   {f.name:<38} {f.stat().st_size / 1024:>9.1f} KB')

Writing artifacts to temp directory...

✅ Artifacts saved to /Users/ayoadeabraham/PycharmProjects/silver-screen/backend/artifacts
   cosine_topk_index.json                  124914.9 KB
   index_to_movie_id.json                     664.0 KB
   kmeans.pkl                                3476.3 KB
   movie_ml.parquet                          1534.7 KB
   movies_metadata.parquet                  23731.1 KB
   scaler.pkl                                   0.7 KB
   tfidf_matrix.npz                          9418.7 KB
   tfidf_vectorizer.pkl                      1521.9 KB
   title_to_index.json                       1565.4 KB


In [13]:
# ── Optional: copy artifacts to Google Drive ───────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive', force_remount=False)
# import shutil
# drive_dest = Path('/content/drive/MyDrive/cinescope_artifacts')
# if drive_dest.exists():
#     shutil.rmtree(drive_dest)
# shutil.copytree(str(ARTIFACTS_DIR), str(drive_dest))
# print(f'✅ Artifacts copied to Google Drive: {drive_dest}')

print('(Drive backup cell — uncomment to activate)')

(Drive backup cell — uncomment to activate)


## ⑨ Serving Functions

In [15]:
class CineScopeRecommender:
    """
    Stateless recommender that loads all persisted artifacts once at startup.
    Compatible with the MovieLens MovieData schema.
    """

    def __init__(self, artifacts_dir: str = 'artifacts'):
        art = Path(artifacts_dir)
        print(f'Loading artifacts from {art.resolve()} ...')

        self.tfidf        = joblib.load(art / 'tfidf_vectorizer.pkl')
        self.tfidf_matrix = sp.load_npz(str(art / 'tfidf_matrix.npz'))
        self.kmeans       = joblib.load(art / 'kmeans.pkl')
        self.scaler       = joblib.load(art / 'scaler.pkl')
        self.meta_df      = pd.read_parquet(art / 'movies_metadata.parquet')
        self.ml_df        = pd.read_parquet(art / 'movie_ml.parquet')

        with open(art / 'cosine_topk_index.json') as f:
            self.cosine_topk = {int(k): v for k, v in json.load(f).items()}

        with open(art / 'title_to_index.json') as f:
            self.title_to_index = json.load(f)

        with open(art / 'index_to_movie_id.json') as f:
            self.index_to_movie_id = {int(k): v for k, v in json.load(f).items()}

        self.movie_id_to_index = {v: k for k, v in self.index_to_movie_id.items()}
        print(f'✅ Ready — {len(self.meta_df):,} movies loaded')

    # ── Lookup helpers ─────────────────────────────────────────────────────────
    def _resolve_index(self, title_or_id):
        """Accept a movie title (str) or movie_id (int) → DataFrame row index."""
        if isinstance(title_or_id, str):
            if title_or_id not in self.title_to_index:
                raise KeyError(f'Title not found: "{title_or_id}"')
            return self.title_to_index[title_or_id]
        else:
            if title_or_id not in self.movie_id_to_index:
                raise KeyError(f'movie_id not found: {title_or_id}')
            return self.movie_id_to_index[title_or_id]

    def _row(self, movie_idx: int) -> dict:
        """Return metadata dict for a given row index."""
        return self.meta_df.iloc[movie_idx].to_dict()

    # ── Public API ─────────────────────────────────────────────────────────────
    def search(self, query: str, limit: int = 10) -> pd.DataFrame:
        """Case-insensitive partial title search, sorted by rating_count (popularity proxy)."""
        q    = query.lower()
        mask = self.meta_df['title'].str.lower().str.contains(q, na=False)
        cols = ['movie_id', 'movieId', 'title', 'avg_rating', 'rating_count', 'genres_list']
        existing = [c for c in cols if c in self.meta_df.columns]
        return (
            self.meta_df[mask][existing]
            .sort_values('rating_count', ascending=False)
            .head(limit)
            .reset_index(drop=True)
        )

    def get_movie(self, title_or_id) -> dict:
        """Full metadata for a single movie by title or movie_id."""
        return self._row(self._resolve_index(title_or_id))

    def similar_movies(self, title_or_id, limit: int = 10) -> pd.DataFrame:
        """Top-N cosine-similar movies based on TF-IDF (genres + tags + title)."""
        idx       = self._resolve_index(title_or_id)
        neighbors = self.cosine_topk.get(idx, [])[:limit]

        rows = []
        for nb_idx, score in neighbors:
            r = self._row(nb_idx)
            rows.append({
                'movie_id':     r['movie_id'],
                'title':        r['title'],
                'similarity':   round(score, 4),
                'avg_rating':   r.get('avg_rating', 0),
                'rating_count': r.get('rating_count', 0),
                'genres':       r.get('genres_list', []),
                'cluster':      r.get('cluster', -1),
            })
        return pd.DataFrame(rows)

    def cluster_feed(self, title_or_id, limit: int = 20,
                     sort: str = 'silhouette') -> pd.DataFrame:
        """
        Movies in the same K-Means cluster as the input movie.
        sort: 'silhouette' | 'avg_rating' | 'rating_count'
        """
        idx     = self._resolve_index(title_or_id)
        src_row = self._row(idx)
        src_id  = src_row['movie_id']
        cluster = src_row['cluster']

        peers = self.meta_df[
            (self.meta_df['cluster'] == cluster) &
            (self.meta_df['movie_id'] != src_id)
        ].copy()

        valid_sorts = {'silhouette', 'avg_rating', 'rating_count'}
        sort_col    = sort if sort in valid_sorts else 'silhouette'
        peers       = peers.sort_values(sort_col, ascending=False)

        cols     = ['movie_id', 'title', 'silhouette', 'cluster', 'avg_rating', 'rating_count', 'genres_list']
        existing = [c for c in cols if c in peers.columns]
        return peers[existing].head(limit).reset_index(drop=True)

    def browse_cluster(self, cluster_id: int, limit: int = 50,
                       sort: str = 'rating_count') -> pd.DataFrame:
        """All movies in a given cluster, sorted by rating_count or avg_rating."""
        subset   = self.meta_df[self.meta_df['cluster'] == cluster_id].copy()
        sort_col = sort if sort in self.meta_df.columns else 'rating_count'
        subset   = subset.sort_values(sort_col, ascending=False)
        cols     = ['movie_id', 'title', 'avg_rating', 'rating_count', 'silhouette', 'genres_list']
        existing = [c for c in cols if c in subset.columns]
        return subset[existing].head(limit).reset_index(drop=True)


# Instantiate (simulates API startup)
rec = CineScopeRecommender()

Loading artifacts from /Users/ayoadeabraham/PycharmProjects/silver-screen/artifacts ...
✅ Ready — 43,884 movies loaded


## ⑩ Interactive Demo

In [16]:
# ── 10a. Search ────────────────────────────────────────────────────────────────
print('=== SEARCH: "toy" ===')
results = rec.search('toy', limit=8)
display(results)

=== SEARCH: "toy" ===


,movie_id,movieId,title,avg_rating,rating_count,genres_list
0,415,1,Toy Story (1995),3.8974,68997,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,1287,3114,Toy Story 2 (1999),3.8120,32683,"[Adventure, Animation, Children, Comedy, Fantasy]"
2,3080,78499,Toy Story 3 (2010),3.8276,20327,"[Adventure, Animation, Children, Comedy, Fanta..."
3,1850,2253,Toys (1992),2.6865,4025,"[Comedy, Fantasy]"
4,6695,201588,Toy Story 4 (2019),3.5890,2802,"[Adventure, Animation, Children, Comedy]"
5,5522,2017,Babes in Toyland (1961),2.9554,1043,"[Children, Fantasy, Musical]"
6,2389,4929,"Toy, The (1982)",2.7601,819,[Comedy]
7,2561,5843,Toy Soldiers (1991),3.0358,797,"[Action, Drama]"


In [19]:
# ── 10b. Similar Movies (Cosine TF-IDF) ───────────────────────────────────────
# MovieLens titles include the year — use the exact title from your dataset
QUERY_MOVIE = 'Toy Story (1995)'   # ← change to any title in your dataset

print(f'=== SIMILAR MOVIES (cosine TF-IDF): "{QUERY_MOVIE}" ===')
similar = rec.similar_movies(QUERY_MOVIE, limit=10)
display(similar)

=== SIMILAR MOVIES (cosine TF-IDF): "Toy Story (1995)" ===


,movie_id,title,similarity,avg_rating,rating_count,genres,cluster
0,1287,Toy Story 2 (1999),0.8843,3.8120,32683,"[Adventure, Animation, Children, Comedy, Fantasy]",2
1,1283,"Bug's Life, A (1998)",0.8171,3.5581,26736,"[Adventure, Animation, Children, Comedy]",4
2,600,Finding Nemo (2003),0.7232,3.8165,46128,"[Adventure, Animation, Children, Comedy]",2
3,3080,Toy Story 3 (2010),0.7133,3.8276,20327,"[Adventure, Animation, Children, Comedy, Fanta...",4
4,278,"Monsters, Inc. (2001)",0.6949,3.8374,46036,"[Adventure, Animation, Children, Comedy, Fantasy]",2
5,1248,"Incredibles, The (2004)",0.6928,3.8482,41463,"[Action, Adventure, Animation, Children, Comedy]",2
6,4167,Finding Dory (2016),0.6877,3.5376,5624,"[Adventure, Animation, Comedy]",5
7,4111,Monsters University (2013),0.6829,3.4627,6587,"[Adventure, Animation, Comedy]",5
8,694,Cars (2006),0.6818,3.3147,11703,"[Animation, Children, Comedy]",4
9,16559,Knick Knack (1989),0.6554,3.4769,347,"[Animation, Children]",5


In [20]:
# ── 10c. Cluster Discovery Feed ────────────────────────────────────────────────
print(f'=== CLUSTER FEED (silhouette sort): "{QUERY_MOVIE}" ===')
feed = rec.cluster_feed(QUERY_MOVIE, limit=10, sort='silhouette')
display(feed)

print(f'\n=== CLUSTER FEED (rating_count sort): "{QUERY_MOVIE}" ===')
feed2 = rec.cluster_feed(QUERY_MOVIE, limit=10, sort='rating_count')
display(feed2)

=== CLUSTER FEED (silhouette sort): "Toy Story (1995)" ===


,movie_id,title,silhouette,cluster,avg_rating,rating_count,genres_list
0,4,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.676602,2,3.9102,55275,"[Mystery, Sci-Fi, Thriller]"
1,957,Good Will Hunting (1997),0.674860,2,4.0911,50824,"[Drama, Romance]"
2,173,"Fugitive, The (1993)",0.668116,2,3.9699,57073,[Thriller]
3,103,Saving Private Ryan (1998),0.660150,2,4.0487,58368,"[Action, Drama, War]"
4,216,Independence Day (a.k.a. ID4) (1996),0.655649,2,3.3811,57224,"[Action, Adventure, Sci-Fi, Thriller]"
5,369,"Dark Knight, The (2008)",0.652210,2,4.1733,59334,"[Action, Crime, Drama, IMAX]"
6,476,"Truman Show, The (1998)",0.631165,2,3.8918,44140,"[Comedy, Drama, Sci-Fi]"
7,322,Seven (a.k.a. Se7en) (1995),0.618462,2,4.0872,63298,"[Mystery, Thriller]"
8,399,Ocean's Eleven (2001),0.584149,2,3.8134,41217,"[Crime, Thriller]"
9,280,"Beautiful Mind, A (2001)",0.582757,2,3.9586,41139,"[Drama, Romance]"



=== CLUSTER FEED (rating_count sort): "Toy Story (1995)" ===


,movie_id,title,silhouette,cluster,avg_rating,rating_count,genres_list
0,161,"Shawshank Redemption, The (1994)",0.0,2,4.4046,102929,"[Crime, Drama]"
1,21,Forrest Gump (1994),0.0,2,4.0527,100296,"[Comedy, Drama, Romance, War]"
2,159,Pulp Fiction (1994),0.0,2,4.1970,98409,"[Comedy, Crime, Drama, Thriller]"
3,514,"Matrix, The (1999)",0.0,2,4.1564,93808,"[Action, Sci-Fi, Thriller]"
4,25,"Silence of the Lambs, The (1991)",0.0,2,4.1484,90330,"[Crime, Horror, Thriller]"
5,15,Star Wars: Episode IV - A New Hope (1977),0.0,2,4.0998,85010,"[Action, Adventure, Sci-Fi]"
6,356,Fight Club (1999),0.0,2,4.2288,77332,"[Action, Crime, Drama, Thriller]"
7,209,Jurassic Park (1993),0.0,2,3.6986,75233,"[Action, Adventure, Sci-Fi, Thriller]"
8,22,Schindler's List (1993),0.0,2,4.2370,73849,"[Drama, War]"
9,339,"Lord of the Rings: The Fellowship of the Ring,...",0.0,2,4.0921,73122,"[Adventure, Fantasy]"


In [21]:
# ── 10d. Browse a specific cluster ─────────────────────────────────────────────
movie_info = rec.get_movie(QUERY_MOVIE)
cluster_id = int(movie_info['cluster'])

print(f'=== ALL MOVIES IN CLUSTER {cluster_id} (sorted by avg_rating) ===')
cluster_browse = rec.browse_cluster(cluster_id, limit=20, sort='avg_rating')
display(cluster_browse)

=== ALL MOVIES IN CLUSTER 2 (sorted by avg_rating) ===


,movie_id,title,avg_rating,rating_count,silhouette,genres_list
0,161,"Shawshank Redemption, The (1994)",4.4046,102929,0.00000,"[Crime, Drama]"
1,218,"Godfather, The (1972)",4.3170,66440,0.00000,"[Crime, Drama]"
2,373,"Usual Suspects, The (1995)",4.2651,67750,0.00000,"[Crime, Mystery, Thriller]"
3,64,"Godfather: Part II, The (1974)",4.2645,43111,0.00000,"[Crime, Drama]"
4,22,Schindler's List (1993),4.2370,73849,0.00000,"[Drama, War]"
5,356,Fight Club (1999),4.2288,77332,0.00000,"[Action, Crime, Drama, Thriller]"
6,2532,Spirited Away (Sen to Chihiro no kamikakushi) ...,4.2131,33333,0.00000,"[Adventure, Animation, Fantasy]"
7,1222,One Flew Over the Cuckoo's Nest (1975),4.2042,44592,0.00000,[Drama]
8,1193,Dr. Strangelove or: How I Learned to Stop Worr...,4.2027,32830,0.00000,"[Comedy, War]"
9,159,Pulp Fiction (1994),4.1970,98409,0.00000,"[Comedy, Crime, Drama, Thriller]"


In [22]:
# ── 10e. Full movie detail card ─────────────────────────────────────────────────
movie = rec.get_movie(QUERY_MOVIE)

genres_str = ', '.join(movie.get('genres_list', [])) or '—'
tags_str   = ', '.join(movie.get('tags_list', [])[:8]) or '—'

html = f"""
<div style="font-family:monospace;background:#111;color:#eee;padding:20px;
             border-radius:10px;border:1px solid #333;max-width:640px">
  <div style="font-size:1.4em;color:#e8b84b;font-weight:bold;margin-bottom:4px">
    🎬 {movie.get('title', '')}
  </div>
  <div style="color:#888;font-size:0.85em;margin-bottom:14px">
    {int(movie.get('year', 0))} &nbsp;·&nbsp; MovieLens ID: {movie.get('movieId', '—')}
    &nbsp;·&nbsp; IMDB: {movie.get('imdbId', '—')}
  </div>
  <table style="border-collapse:collapse;width:100%">
    <tr><td style="color:#888;padding:4px 12px 4px 0;vertical-align:top">Genres</td>
        <td style="color:#eee">{genres_str}</td></tr>
    <tr><td style="color:#888;padding:4px 12px 4px 0;vertical-align:top">Top Tags</td>
        <td style="color:#aaa;font-size:0.9em">{tags_str}</td></tr>
    <tr><td style="color:#888;padding:4px 12px 4px 0">Avg Rating</td>
        <td style="color:#2ecc71">⭐ {movie.get('avg_rating', 0):.2f} / 5.0
            &nbsp;({int(movie.get('rating_count', 0)):,} ratings)</td></tr>
    <tr><td style="color:#888;padding:4px 12px 4px 0">Cluster</td>
        <td style="color:#9b59b6">{movie.get('cluster', '—')}</td></tr>
    <tr><td style="color:#888;padding:4px 12px 4px 0">Silhouette</td>
        <td style="color:#3498db">{round(float(movie.get('silhouette', 0)), 4)}</td></tr>
  </table>
</div>
"""
display(HTML(html))

Genres,"Adventure, Animation, Children, Comedy, Fantasy"
Top Tags,"children, disney, animation, children, disney, disney, pixar, animation"
Avg Rating,"⭐ 3.90 / 5.0 (68,997 ratings)"
Cluster,2
Silhouette,0.0


## ⑪ Cluster Summary Report

In [23]:
summary = []
for c in sorted(rec.meta_df['cluster'].unique()):
    g          = rec.meta_df[rec.meta_df['cluster'] == c]
    best_title = g.loc[g['silhouette'].idxmax(), 'title']

    all_genres = []
    for gl in g['genres_list'].dropna():
        if isinstance(gl, list):
            all_genres.extend(gl)
    top_genres = pd.Series(all_genres).value_counts().index[:3].tolist()

    summary.append({
        'cluster':        c,
        'n_movies':       len(g),
        'top_genres':     ', '.join(top_genres),
        'avg_rating':     round(g['avg_rating'].mean(), 3),
        'avg_sil':        round(g['silhouette'].mean(), 4),
        'best_fit_movie': best_title,
    })

summary_df = pd.DataFrame(summary).set_index('cluster')
print('=== CLUSTER SUMMARY REPORT ===')
display(summary_df)

=== CLUSTER SUMMARY REPORT ===


,n_movies,top_genres,avg_rating,avg_sil,best_fit_movie
cluster,,,,,
0,15706,,3.208,0.0100,Bopha! (1993)
1,4060,,1.987,0.0200,Komodo vs. Cobra (2005)
2,137,,3.858,0.0619,Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
3,165,,3.061,0.0911,Falling Skies
4,907,,3.556,0.0372,10 Things I Hate About You (1999)
5,11844,,3.684,0.0117,Time Indefinite (1993)
6,11065,,2.703,0.0123,Daisy-Head Mayzie (1995)


In [24]:
# ── Quick reload test (simulate API restart) ────────────────────────────────────
print('Simulating API restart — reloading all artifacts from disk...')
rec2 = CineScopeRecommender()
# Use the first movie in the dataset to verify reload works
test_title = rec2.meta_df.iloc[0]['title']
test       = rec2.similar_movies(test_title, limit=5)
print(f'\nSimilar to "{test_title}":')
display(test)
print('✅ Reload test passed')

Simulating API restart — reloading all artifacts from disk...
Loading artifacts from /Users/ayoadeabraham/PycharmProjects/silver-screen/artifacts ...
✅ Ready — 43,884 movies loaded

Similar to "Sense and Sensibility (1995)":


,movie_id,title,similarity,avg_rating,rating_count,genres,cluster
0,30,Emma (1996),0.6678,3.7704,8314,"[Comedy, Drama, Romance]",4
1,3744,Persuasion (1995),0.6483,4.0412,3310,"[Drama, Romance]",5
2,9023,Northanger Abbey (2007),0.6217,3.7121,264,"[Drama, Romance]",5
3,13303,Persuasion (2007),0.5978,3.8444,408,"[Drama, Romance]",5
4,6386,Pride and Prejudice (1940),0.5927,3.6475,610,"[Comedy, Drama, Romance]",5


✅ Reload test passed
